# Notebook to demo `QR_AsGeom` function.

The function `qr_asgeom(qr,cell,dist=0.0)` converts a qr code into a rectangular geometry object.

In [21]:
import os

import geofunctions as S
import geopandas as gp
import pyspark.sql.functions as F
import xyzservices.providers as xyz

In [26]:
cell = 2.0  # QR in degrees.
dist = 0.00001  # Padding in degrees.
wkid = 4326  # SR ID

In [27]:
df = (
    spark.read.format("shp")
    .options(path="../data/world/tz_world.shp")
    .load()
    .filter(F.col("tzid").startswith("America/New_York"))
    # Multipart to singlepart.
    .select(
        S.st_dump_explode("shape").alias("shape"),
    )
    # Convert each part to a list of qr's.
    .select(
        S.qr_list_explode(
            "shape",
            cell,
            dist,
        ).alias("qr"),
    )
    .dropDuplicates(["qr"])
    # Convert the qr to a geom for visualization.
    .select(
        F.hex("qr").alias("qr"),
        S.qr_asgeom("qr", cell).alias("geometry"),
    )
)

In [28]:
pdf = df.toPandas()
pdf.geometry = gp.GeoSeries.from_wkb(pdf.geometry)
gdf = gp.GeoDataFrame(pdf, crs=f"EPSG:{wkid}")
# Enable the spatial index.
gdf.sindex
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)